In [43]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/mri-datasets/NYU-1_sMRI_AAL.csv
/kaggle/input/mri-datasets/Brown_TestRelease_phenotypic.csv
/kaggle/input/mri-datasets/NYU_phenotypic.csv
/kaggle/input/mri-datasets/Pittsburgh_phenotypic.csv
/kaggle/input/mri-datasets/OHSU_sMRI_AAL.csv
/kaggle/input/mri-datasets/OHSU_phenotypic.csv
/kaggle/input/mri-datasets/Peking_1_sMRI_AAL.csv
/kaggle/input/mri-datasets/Brown_sMRI_AAL.csv
/kaggle/input/mri-datasets/Pittsburgh_sMRI_AAL.csv
/kaggle/input/mri-datasets/Peking_1_phenotypic.csv
/kaggle/input/mri-datasets/NeuroIMAGE_phenotypic.csv
/kaggle/input/mri-datasets/NeuroIMAGE_SUBJECTS_sMRI_AAL.csv
/kaggle/input/mri-datasets/KKI_sMRI_AAL.csv
/kaggle/input/mri-datasets/NYU-2_sMRI_AAL.csv
/kaggle/input/mri-datasets/KKI_phenotypic.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sMRI_files = {
    'NYU-1':'/kaggle/input/mri-datasets/NYU-1_sMRI_AAL.csv',
    'NYU-2': '/kaggle/input/mri-datasets/NYU-2_sMRI_AAL.csv',
    'OHSU': '/kaggle/input/mri-datasets/OHSU_sMRI_AAL.csv',
    'Pittsburgh': '/kaggle/input/mri-datasets/Pittsburgh_sMRI_AAL.csv',
    'Peking_1': '/kaggle/input/mri-datasets/Peking_1_sMRI_AAL.csv',
    'NeuroIMAGE': '/kaggle/input/mri-datasets/NeuroIMAGE_SUBJECTS_sMRI_AAL.csv',
    'KKI': '/kaggle/input/mri-datasets/KKI_sMRI_AAL.csv'
}

pheno_files = {
    'NYU-1': '/kaggle/input/mri-datasets/NYU_phenotypic.csv',
    'NYU-2': '/kaggle/input/mri-datasets/NYU_phenotypic.csv',
    'OHSU': '/kaggle/input/mri-datasets/OHSU_phenotypic.csv',
    'Pittsburgh': '/kaggle/input/mri-datasets/Pittsburgh_phenotypic.csv',
    'Peking_1': '/kaggle/input/mri-datasets/Peking_1_phenotypic.csv',
    'NeuroIMAGE': '/kaggle/input/mri-datasets/NeuroIMAGE_phenotypic.csv',
    'KKI': '/kaggle/input/mri-datasets/KKI_phenotypic.csv'
}

all_sites = []

for site in sMRI_files.keys():
    print(f"Processing site: {site}")

    df_sMRI = pd.read_csv(sMRI_files[site])
    df_pheno = pd.read_csv(pheno_files[site])

   
    df_sMRI['sub_id'] = df_sMRI['Subject'].str.extract(r'sub-(\d+)_')[0]

    df_pheno['ScanDir ID'] = df_pheno['ScanDir ID'].astype(str)

    dx_dict = dict(zip(df_pheno['ScanDir ID'], df_pheno['DX']))

    dx_list = []
    for sub in df_sMRI['sub_id']:
        dx_val = np.nan
        for pid, dx in dx_dict.items():
            if pid.endswith(sub.lstrip("0")):
                dx_val = dx
                break
        dx_list.append(dx_val)

    df_sMRI['DX'] = dx_list
    df_sMRI['site'] = site

    all_sites.append(df_sMRI)

final_df = pd.concat(all_sites, ignore_index=True)
print("Total subjects before cleaning:", len(final_df))


Processing site: NYU-1
Processing site: NYU-2
Processing site: OHSU
Processing site: Pittsburgh
Processing site: Peking_1
Processing site: NeuroIMAGE
Processing site: KKI
Total subjects before cleaning: 766


In [ ]:
final_df = final_df.dropna(subset=['DX'])

final_df = final_df[final_df['DX'] != 'pending']

final_df['DX'] = pd.to_numeric(final_df['DX'], errors='coerce')
final_df = final_df.dropna(subset=['DX'])

final_df['DX'] = final_df['DX'].astype(int)

final_df['DX_binary'] = (final_df['DX'] != 0).astype(int)

print("DX distribution:")
print(final_df['DX_binary'].value_counts())


DX distribution:
DX_binary
0    375
1    231
Name: count, dtype: int64


In [46]:
le = LabelEncoder()
final_df['site'] = le.fit_transform(final_df['site'])

print("Site encoding:", dict(zip(le.classes_, le.transform(le.classes_))))


Site encoding: {'KKI': np.int64(0), 'NYU-1': np.int64(1), 'NYU-2': np.int64(2), 'NeuroIMAGE': np.int64(3), 'OHSU': np.int64(4), 'Peking_1': np.int64(5), 'Pittsburgh': np.int64(6)}


In [ ]:
final_df = final_df.drop_duplicates().reset_index(drop=True)

drop_cols = ['Subject', 'sub_id', 'DX']
X = final_df.drop(columns=drop_cols + ['DX_binary'])
y = final_df['DX_binary']

print("Final X shape:", X.shape)
print("Final y shape:", y.shape)


Final X shape: (606, 117)
Final y shape: (606,)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import torch
import pandas as pd
import numpy as np

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights_dict = dict(zip(classes, class_weights))
print("Class weights:", class_weights_dict)


n_neg = np.sum(y_train == 0)
n_pos = np.sum(y_train == 1)

pos_weight = torch.tensor(n_neg / n_pos, dtype=torch.float32)
print("pos_weight for BCEWithLogitsLoss:", pos_weight)


print("\nTrain distribution:\n", pd.Series(y_train).value_counts())
print("\nValidation distribution:\n", pd.Series(y_val).value_counts())
print("\nTest distribution:\n", pd.Series(y_test).value_counts())


print("\nShapes:")
print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val:", X_val.shape, "| y_val:", y_val.shape)
print("X_test:", X_test.shape, "| y_test:", y_test.shape)

Class weights: {np.int64(0): np.float64(0.8066666666666666), np.int64(1): np.float64(1.315217391304348)}
pos_weight for BCEWithLogitsLoss: tensor(1.6304)

Train distribution:
 DX_binary
0    225
1    138
Name: count, dtype: int64

Validation distribution:
 DX_binary
0    75
1    46
Name: count, dtype: int64

Test distribution:
 DX_binary
0    75
1    47
Name: count, dtype: int64

Shapes:
X_train: (363, 117) | y_train: (363,)
X_val: (121, 117) | y_val: (121,)
X_test: (122, 117) | y_test: (122,)


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np
import pandas as pd


class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

train_dataset = TabularDataset(pd.DataFrame(X_train_scaled), y_train)
test_dataset  = TabularDataset(pd.DataFrame(X_test_scaled), y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)


class FTTransformer(nn.Module):
    def __init__(self, input_dim, d_model=128, n_heads=4, n_layers=3, dropout=0.15):
        super().__init__()
        self.embedding = nn.Linear(input_dim, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, 1)

    def forward(self, x):
        x = self.embedding(x).unsqueeze(1)
        x = self.transformer(x)
        x = x.squeeze(1)
        x = self.dropout(x)
        return self.classifier(x)


device = "cuda" if torch.cuda.is_available() else "cpu"
model = FTTransformer(input_dim=X_train.shape[1]).to(device)

raw_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
pos_weight = torch.tensor([raw_pos_weight * 0.7], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-5
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)


num_epochs = 80
patience = 8
best_auc = 0
counter = 0

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device).unsqueeze(1)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    total_loss /= len(train_loader.dataset)

    model.eval()
    probs, labels = [], []

    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            out = torch.sigmoid(model(xb)).cpu().numpy()
            probs.extend(out)
            labels.extend(yb.numpy())

    probs = np.array(probs).reshape(-1)
    labels = np.array(labels)

    roc = roc_auc_score(labels, probs)
    preds_05 = (probs > 0.5).astype(int)
    acc_05 = accuracy_score(labels, preds_05)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Loss: {total_loss:.4f} | "
        f"Acc0.5: {acc_05:.4f} | "
        f"ROC-AUC: {roc:.4f}"
    )

    scheduler.step(roc)

    if roc > best_auc:
        best_auc = roc
        counter = 0
        torch.save(model.state_dict(), "best_ft_transformer.pt")
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break


model.load_state_dict(torch.load("best_ft_transformer.pt"))
model.eval()

probs = []
with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(device)
        probs.extend(torch.sigmoid(model(xb)).cpu().numpy())

probs = np.array(probs).reshape(-1)
labels = np.array(y_test)

best_acc = 0
best_t = 0

for t in np.linspace(0.1, 0.9, 81):
    preds = (probs > t).astype(int)
    acc = accuracy_score(labels, preds)
    if acc > best_acc:
        best_acc = acc
        best_t = t

final_roc = roc_auc_score(labels, probs)

print("\n===== FINAL RESULTS =====")
print(f"Best Threshold : {best_t:.2f}")
print(f"Final Accuracy : {best_acc:.4f}")
print(f"Final ROC-AUC  : {final_roc:.4f}")


Epoch [1/80] | Loss: 0.7532 | Acc0.5: 0.6803 | ROC-AUC: 0.7413
Epoch [2/80] | Loss: 0.6357 | Acc0.5: 0.6721 | ROC-AUC: 0.7611
Epoch [3/80] | Loss: 0.6327 | Acc0.5: 0.7049 | ROC-AUC: 0.7535
Epoch [4/80] | Loss: 0.5683 | Acc0.5: 0.6721 | ROC-AUC: 0.7359
Epoch [5/80] | Loss: 0.5723 | Acc0.5: 0.6393 | ROC-AUC: 0.7288
Epoch [6/80] | Loss: 0.5329 | Acc0.5: 0.7295 | ROC-AUC: 0.8017
Epoch [7/80] | Loss: 0.5361 | Acc0.5: 0.5574 | ROC-AUC: 0.6539
Epoch [8/80] | Loss: 0.4969 | Acc0.5: 0.6885 | ROC-AUC: 0.7705
Epoch [9/80] | Loss: 0.4281 | Acc0.5: 0.6721 | ROC-AUC: 0.7529
Epoch [10/80] | Loss: 0.4064 | Acc0.5: 0.6475 | ROC-AUC: 0.7138
Epoch [11/80] | Loss: 0.3772 | Acc0.5: 0.6639 | ROC-AUC: 0.7322
Epoch [12/80] | Loss: 0.3099 | Acc0.5: 0.6393 | ROC-AUC: 0.7226
Epoch [13/80] | Loss: 0.2993 | Acc0.5: 0.6475 | ROC-AUC: 0.7330
Epoch [14/80] | Loss: 0.2402 | Acc0.5: 0.6721 | ROC-AUC: 0.7220
Early stopping at epoch 14

===== FINAL RESULTS =====
Best Threshold : 0.51
Final Accuracy : 0.7377
Final ROC-AUC